# revT2V — training (Kaggle)

Trains a student method's LoRA weights on Kaggle's free T4/P100 GPU. Kaggle gives ~30 GPU-hours/week split into 12-hour sessions, so this notebook is built to be re-run: it resumes the dataset build and training from wherever a private Hugging Face Hub repo last left off. See `LEARNING.md` for the step-by-step curriculum this notebook assumes you're following.

**Before running:** in Kaggle's notebook settings, turn on a GPU accelerator (Settings -> Accelerator -> GPU T4 x2 or P100), and add your Hugging Face token as a Kaggle Secret named `HF_TOKEN` (Add-ons -> Secrets). You'll also need two private HF Hub repos created ahead of time (or let `huggingface_hub` create them on first push): one `dataset` repo for latent shards, one `model` repo for checkpoints.

## 1. Clone/pull the repo and install it

In [ ]:
!git clone -q https://github.com/silentlooop/revT2V.git /kaggle/working/revT2V 2>/dev/null || git -C /kaggle/working/revT2V pull -q
%cd /kaggle/working/revT2V
!pip install -q -e .


## 2. Confirm the GPU is visible

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU visible — check Settings > Accelerator"
print(torch.__version__, torch.cuda.get_device_name(0))

## 3. Load the HF token from Kaggle Secrets

`revt2v.utils.get_hf_token` also checks this automatically at runtime, but setting `HF_TOKEN` in the environment here means any `huggingface_hub` call (including inside `diffusers`) picks it up too.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient

os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")

## 4. Set your repo ids

Replace these with your own private HF Hub repos once, then leave this notebook as-is across re-runs.

In [ ]:
HF_DATASET_REPO = "silentlooop/revt2v-data"  
HF_CHECKPOINT_REPO = "silentlooop/revt2v-ckpt"  
METHOD = "attn_rotation"  # baseline | attn_rotation | attn_rotation_all | motion_prior

## 5. Build the training dataset (LEARNING.md step 2)

Resumable: shards already on disk or already pushed to `HF_DATASET_REPO` are skipped, so re-running this cell after a session restart continues where it left off instead of regenerating everything.

In [ ]:
!python scripts/build_dataset.py --prompts prompts/train.txt --hf-repo {HF_DATASET_REPO}

## 6. Train (LEARNING.md steps 3, 5, or 6 depending on `METHOD`)

Pulls the latest checkpoint from `HF_CHECKPOINT_REPO` if one exists (so switching Kaggle sessions resumes automatically), and pushes back every few minutes plus once at the end.

In [ ]:
!python scripts/train.py --config configs/{METHOD}.yaml --hf-repo {HF_CHECKPOINT_REPO} --hf-dataset-repo {HF_DATASET_REPO}

## 7. When the 12-hour session is about to end

Nothing to do manually — `train.py` force-pushes a final checkpoint when it exits. If Kaggle kills the session before `num_steps` is reached, just re-run this notebook (steps 1-6): it will resume from the last checkpoint pushed to the Hub.

Once training looks done (check the loss curve above, or pull `configs/<method>.yaml`'s `num_steps` down for a quick first run), head to `colab_test.ipynb` to serve and evaluate the checkpoint.